In [13]:
import numpy as np
import cv2
import time

assets_dir = '../Assets/'

In [14]:
img = cv2.imread(assets_dir + '/generic/Lena.jpg', cv2.IMREAD_GRAYSCALE)
if img is None:
    img = np.random.randint(0, 255, (256, 256), dtype=np.uint8)

h, w = img.shape
print(f"Image dimensions: {h} x {w}")

img_bordered = cv2.copyMakeBorder(img, 2, 2, 2, 2, cv2.BORDER_CONSTANT)

Image dimensions: 512 x 512


In [15]:
kernel1 = np.array([
    [1,  4,  6,  4, 1],
    [4, 16, 24, 16, 4],
    [6, 24, 36, 24, 6],
    [4, 16, 24, 16, 4],
    [1,  4,  6,  4, 1]
], dtype=np.float32)

kernel1 = kernel1 / np.sum(kernel1)

print("Original 2D Kernel:")
print(kernel1)
print()

Original 2D Kernel:
[[0.00390625 0.015625   0.0234375  0.015625   0.00390625]
 [0.015625   0.0625     0.09375    0.0625     0.015625  ]
 [0.0234375  0.09375    0.140625   0.09375    0.0234375 ]
 [0.015625   0.0625     0.09375    0.0625     0.015625  ]
 [0.00390625 0.015625   0.0234375  0.015625   0.00390625]]



In [16]:
def manual_convolution(padded_image, kernel, flip_kernel=True):
    if flip_kernel:
        kernel = np.flip(kernel)
    
    kernel_size = kernel.shape[0]
    
    img_height, img_width = padded_image.shape
    output_height = img_height - kernel_size + 1
    output_width = img_width - kernel_size + 1
    output = np.zeros((output_height, output_width), dtype=np.float32)
    
    operation_count = 0
    
    for i in range(output_height):
        for j in range(output_width):
            roi = padded_image[i:i+kernel_size, j:j+kernel_size]
            output[i, j] = np.sum(roi * kernel)
            # operations: kernel_size^2 multiplications + (kernel_size^2 - 1) additions
            operation_count += kernel_size * kernel_size * 2 - 1
    
    return output, operation_count

In [17]:
def separable_convolution(padded_image, kernel_x, kernel_y):
    temp_result = np.zeros_like(padded_image, dtype=np.float32)
    operation_count = 0
    
    kernel_size = len(kernel_x)
    half_size = kernel_size // 2
    
    # Horizontal convolution
    for i in range(padded_image.shape[0]):
        for j in range(half_size, padded_image.shape[1] - half_size):
            temp_result[i, j] = np.sum(padded_image[i, j-half_size:j+half_size+1] * kernel_x)
            operation_count += kernel_size * 2 - 1
    
    # Vertical convolution
    final_result = np.zeros((padded_image.shape[0] - kernel_size + 1, 
                            padded_image.shape[1] - kernel_size + 1), dtype=np.float32)
    
    for i in range(half_size, temp_result.shape[0] - half_size):
        for j in range(half_size, temp_result.shape[1] - half_size):
            final_result[i-half_size, j-half_size] = np.sum(temp_result[i-half_size:i+half_size+1, j] * kernel_y)
            operation_count += kernel_size * 2 - 1
    
    return final_result, operation_count

In [18]:
# ======================= PART A: 2D Convolution =======================
print("=" * 60)
print("PART A: 2D Convolution")
print("=" * 60)

start_time = time.time()
image_conv_2d, ops_2d = manual_convolution(img_bordered, kernel1)
time_2d = time.time() - start_time

print(f"2D Convolution:")
print(f"Operations: {ops_2d:,}")
print(f"Time: {time_2d:.4f} seconds")

norm_image = cv2.normalize(image_conv_2d, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
cv2.imshow('Normalized Image', norm_image)
cv2.waitKey(0)
cv2.destroyAllWindows()


PART A: 2D Convolution
2D Convolution:
Operations: 12,845,056
Time: 0.7633 seconds


In [19]:
# ======================= PART B: Separable Convolution =======================
print("=" * 60)
print("PART B: Separable Convolution")
print("=" * 60)

kernel_x_ideal = np.array([1, 4, 6, 4, 1], dtype=np.float32)
kernel_y_ideal = np.array([1, 4, 6, 4, 1], dtype=np.float32)

kernel_x_ideal = kernel_x_ideal / np.sum(kernel_x_ideal)
kernel_y_ideal = kernel_y_ideal / np.sum(kernel_y_ideal)

print("Ideal separable kernels (for Gaussian):")
print(f"X kernel: {kernel_x_ideal}")
print(f"Y kernel: {kernel_y_ideal}")

start_time = time.time()
image_conv_sep, ops_sep = separable_convolution(img_bordered, kernel_x_ideal, kernel_y_ideal)
time_sep = time.time() - start_time

print(f"\nSeparable Convolution:")
print(f"Operations: {ops_sep:,}")
print(f"Time: {time_sep:.4f} seconds")
print(f"Speedup: {ops_2d / ops_sep:.2f}x reduction in operations")
print()

norm_sep_img = cv2.normalize(image_conv_sep, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
cv2.imshow('Normalized Separable Image', norm_sep_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

PART B: Separable Convolution
Ideal separable kernels (for Gaussian):
X kernel: [0.0625 0.25   0.375  0.25   0.0625]
Y kernel: [0.0625 0.25   0.375  0.25   0.0625]

Separable Convolution:
Operations: 4,737,024
Time: 1.1994 seconds
Speedup: 2.71x reduction in operations



In [20]:
kernel2 = np.array([
    [1,  5,  6,  4, 1],
    [4, 15, 24, 16, 4],
    [6, 23, 36, 24, 6],
    [4, 16, 28, 16, 4],
    [1,  4,  6, 7, 1]
], dtype=np.float32)

In [21]:
# ======================= PART C: SVD Decomposition =======================
print("=" * 60)
print("PART C: SVD Decomposition")
print("=" * 60)

# SVD to the 2D kernel
U, S, Vt = np.linalg.svd(kernel2, full_matrices=False)

print("Singular values:", S)
print(f"Largest singular value: {S[0]:.6f}")
print()

# singular vectors (rank-1 approximation)
vector_x_svd = U[:, 0] * np.sqrt(S[0])  # Column vector
vector_y_svd = Vt[0, :] * np.sqrt(S[0])  # Row vector

print("SVD-derived vectors:")
print(f"X vector (column): {vector_x_svd}")
print(f"Y vector (row): {vector_y_svd}")

# Reconstruct kernel - rank-1 approximation
kernel_approx = np.outer(vector_x_svd, vector_y_svd)

print("\nOriginal kernel:")
print(kernel1)
print("\nSVD approximated kernel (rank-1):")
print(kernel_approx)


PART C: SVD Decomposition
Singular values: [7.1126938e+01 3.2465906e+00 1.4930035e+00 4.3473145e-01 1.7475761e-16]
Largest singular value: 71.126938

SVD-derived vectors:
X vector (column): [-1.0456492 -3.912585  -5.895879  -4.31464   -1.1609155]
Y vector (row): [-0.99105537 -3.841022   -6.1889772  -4.013187   -0.99105537]

Original kernel:
[[0.00390625 0.015625   0.0234375  0.015625   0.00390625]
 [0.015625   0.0625     0.09375    0.0625     0.015625  ]
 [0.0234375  0.09375    0.140625   0.09375    0.0234375 ]
 [0.015625   0.0625     0.09375    0.0625     0.015625  ]
 [0.00390625 0.015625   0.0234375  0.015625   0.00390625]]

SVD approximated kernel (rank-1):
[[ 1.0362962  4.0163617  6.471499   4.1963854  1.0362962]
 [ 3.8775885 15.028325  24.2149    15.701935   3.8775885]
 [ 5.8431425 22.6462    36.48946   23.661264   5.8431425]
 [ 4.276047  16.572628  26.703209  17.315456   4.276047 ]
 [ 1.1505315  4.459102   7.18488    4.658971   1.1505315]]


In [22]:
# Calculate decomposition error
error_matrix = kernel1 - kernel_approx
absolute_error = np.abs(error_matrix)
max_error = np.max(absolute_error)
mean_error = np.mean(absolute_error)
frobenius_error = np.linalg.norm(error_matrix, 'fro')

print(f"\nDecomposition Error Analysis:")
print(f"Maximum absolute error: {max_error:.8f}")
print(f"Mean absolute error: {mean_error:.8f}")
print(f"Frobenius norm error: {frobenius_error:.8f}")
print(f"Relative error (Frobenius): {frobenius_error / np.linalg.norm(kernel1, 'fro') * 100:.6f}%")

print("\nError matrix:")
print(error_matrix)


Decomposition Error Analysis:
Maximum absolute error: 36.34883499
Mean absolute error: 10.42751217
Frobenius norm error: 70.85388184
Relative error (Frobenius): 25912.277222%

Error matrix:
[[ -1.03239    -4.0007367  -6.4480615  -4.1807604  -1.03239  ]
 [ -3.8619635 -14.965825  -24.12115   -15.639435   -3.8619635]
 [ -5.819705  -22.55245   -36.348835  -23.567514   -5.819705 ]
 [ -4.260422  -16.510128  -26.609459  -17.252956   -4.260422 ]
 [ -1.1466253  -4.443477   -7.1614423  -4.643346   -1.1466253]]


In [23]:
# Apply SVD-based separable convolution
start_time = time.time()
image_conv_svd, ops_svd = separable_convolution(img_bordered, vector_x_svd, vector_y_svd)
time_svd = time.time() - start_time

print(f"\nSVD Separable Convolution:")
print(f"Operations: {ops_svd:,}")
print(f"Time: {time_svd:.4f} seconds")

# ======================= Comparison of Results =======================
print("\n" + "=" * 60)
print("COMPARISON OF ALL METHODS")
print("=" * 60)

print(f"{'Method':<20} {'Operations':<15} {'Time (s)':<10} {'Relative Speed':<15}")
print("-" * 70)
print(f"{'2D Convolution':<20} {ops_2d:<15,} {time_2d:<10.4f} {'1.00x':<15}")
print(f"{'Separable (Ideal)':<20} {ops_sep:<15,} {time_sep:<10.4f} {f'{time_2d/time_sep:.2f}x':<15}")
print(f"{'SVD Separable':<20} {ops_svd:<15,} {time_svd:<10.4f} {f'{time_2d/time_svd:.2f}x':<15}")

norm_svd_img = cv2.normalize(image_conv_svd, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
cv2.imshow('Normalized SVD Image', norm_svd_img)
cv2.waitKey(0)
cv2.destroyAllWindows()



SVD Separable Convolution:
Operations: 4,737,024
Time: 1.2045 seconds

COMPARISON OF ALL METHODS
Method               Operations      Time (s)   Relative Speed 
----------------------------------------------------------------------
2D Convolution       12,845,056      0.7633     1.00x          
Separable (Ideal)    4,737,024       1.1994     0.64x          
SVD Separable        4,737,024       1.2045     0.63x          


In [ ]:
diff_ideal = np.abs(image_conv_2d - image_conv_sep)
diff_svd = np.abs(image_conv_2d - image_conv_svd)

print(f"\nImage Comparison:")
print(f"Max difference (2D vs Ideal Separable): {np.max(diff_ideal):.8f}")
print(f"Max difference (2D vs SVD Separable): {np.max(diff_svd):.8f}")
print(f"Mean difference (2D vs SVD Separable): {np.mean(diff_svd):.8f}")


Image Comparison:
Max difference (2D vs Ideal Separable): 0.00000000
Max difference (2D vs SVD Separable): 60428.26171875
Mean difference (2D vs SVD Separable): 32285.91015625
